### Imports and Configuration

#### Shared Blackboard Multi-Agent Insider Threat Detection


Key innovation: a Shared Blackboard (message bus) enables inter-agent communication:

  - Agent 1 (CAE) → writes scores + dynamic threshold + high-anomaly flag
  - Agent 2 (DistilBERT) → reads CAE flag, boosts score for flagged sessions
  - Agent 3 (Temporal) → reads upstream flags, uses global baselines for flagged sessions
  - Agent 4 (Orchestrator) → reads all scores + evidence features from blackboard

Architecture:
  - Agent 1 (CAE)       → "How statistically abnormal is this session?" (general anomaly)
  - Agent 2 (DistilBERT) → "How suspicious is the text content?" (MPNet Scanner + DistilBERT Classifier, with CAE-guided boost)
  - Agent 3 (Temporal)  → "How abnormal is this FOR THIS USER?" (context-aware)
  - Agent 4 (IF Orch.)  → Non-linear fusion of scores + inter-agent evidence

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pickle
from tqdm.auto import tqdm
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries Imported")

Libraries Imported


In [2]:
CONFIG = {
    "dataset_path": "../Dataset/r6.2/",
    "processed_file": "../DatasetsProcessed/data_exfiltration_content_dataset_updated.csv",
    "agent1_path": "../SavedModels/v10_agent1_cae.pth",
    "agent2_model_dir": "../SavedModels/final_insider_threat_model",
    "agent2_scanner_model": "all-mpnet-base-v2",
    "agent3_path": "../SavedModels/v10_agent3_baselines.pkl",
    "agent4_path": "../SavedModels/v10_agent4_orchestrator.pkl",
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

# --- Blackboard Configuration (tuneable) ---
CAE_THRESHOLD_SIGMA = 2.0      # mean + N*std for CAE dynamic threshold
NLP_BOOST_MULTIPLIER = 1.5     # boost factor for NLP when CAE flags high
NLP_FLAG_PERCENTILE = 90       # percentile above which NLP is considered "high"

print(f"Running on device: {CONFIG['device']}")
print("Configuration loaded")

Running on device: cuda
Configuration loaded


### Shared Blackboard: Inter-Agent Communication Hub

In [3]:
class Blackboard:
    """
    Shared communication hub for the multi-agent system.
    Agents write their scores, thresholds, and flags here.
    Downstream agents read upstream results to adapt their behavior.
    """
    def __init__(self, n_samples):
        self.n_samples = n_samples
        self.data = {}               # Storage for per-session arrays / scalars
        self.messages = []           # Ordered list of inter-agent messages

    def write(self, agent_name, key, value, msg=None):
        """Write a per-session array or scalar to the blackboard."""
        full_key = f"{agent_name}:{key}"
        self.data[full_key] = value
        if msg:
            self.messages.append({
                'agent': agent_name,
                'key': key,
                'msg': msg
            })
            print(f"  [Blackboard] {msg}")

    def read(self, agent_name, key):
        """Read a value from the blackboard. Returns None if not found."""
        return self.data.get(f"{agent_name}:{key}", None)

    def get_flag(self, agent_name, key):
        """Read a boolean flag array from the blackboard."""
        arr = self.read(agent_name, key)
        if arr is None:
            return np.zeros(self.n_samples, dtype=bool)
        return arr.astype(bool)

    def get_messages(self):
        """Return all inter-agent messages (for logging / orchestrator evidence)."""
        return list(self.messages)

    def summary(self):
        """Print a summary of all blackboard contents."""
        print(f"\n{'='*60}")
        print(f"  Blackboard Summary ({len(self.data)} entries, {len(self.messages)} messages)")
        print(f"{'='*60}")
        for key, val in self.data.items():
            if isinstance(val, np.ndarray):
                print(f"  {key}: ndarray shape={val.shape}, dtype={val.dtype}")
            else:
                print(f"  {key}: {type(val).__name__} = {val}")
        print()
        for m in self.messages:
            print(f"  [{m['agent']}] {m['msg']}")
        print(f"{'='*60}\n")

### Data Loading Funtion

In [4]:
def load_cert_data(path):
    data = {}
    # List of expected files in the CERT r6.2 dataset directory
    files = ['insiders.csv', 'logon.csv', 'device.csv', 'file.csv', 'email.csv', 'http.csv']
    
    print("Loading datasets...")
    for f in tqdm(files):
        file_path = os.path.join(path, f)
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            # Clean column namess
            df.columns = [c.strip() for c in df.columns]
            
            # Parse Dates
            date_cols = ['date'] if 'date' in df.columns else ['start', 'end']
            for dc in date_cols:
                if dc in df.columns:
                    df[dc] = pd.to_datetime(df[dc], format='%m/%d/%Y %H:%M:%S', errors='coerce')
            
            key = f.split('.')[0]
            data[key] = df
        else:
            print(f"Warning: {f} not found in {path}")
    return data

### Session Creation and Feature Extraction Function

In [5]:
def create_sessions_and_features(data):
    print("Processing Sessions and Extracting Features...")
        
    logon_df = data['logon'].sort_values(['user', 'date'])
    email_df = data['email'].sort_values('date')
    http_df = data['http'].sort_values('date')
    file_df = data['file'].sort_values('date')
    device_df = data['device'].sort_values('date')
    insiders_df = data['insiders']
    
    # Pre-indexing for speed
    email_df = email_df.set_index('date').sort_index()
    http_df = http_df.set_index('date').sort_index()
    file_df = file_df.set_index('date').sort_index()
    device_df = device_df.set_index('date').sort_index()
    
    # --- 1. PRE-PROCESS THREATS (The "Generic" but Optimized Approach) ---
    # We create a dictionary for O(1) lookup speed.
    # We FILTER for Scenarios 1, 2, 4, 5 (Data Exfiltration Scope)
    # Scenario 3 is usually IT Sabotage/Keylogging, so we skip it.
    
    # Ensure 'scenario' column exists and filter
    target_scenarios = [1, 2, 4, 5]
    if 'scenario' in insiders_df.columns:
        threats_filtered = insiders_df[insiders_df['scenario'].isin(target_scenarios)]
    else:
        threats_filtered = insiders_df # Fallback if column missing

    malicious_map = {}
    for _, row in threats_filtered.iterrows():
        u = row['user']
        if u not in malicious_map: malicious_map[u] = []
        malicious_map[u].append((row['start'], row['end']))
    
    print(f"Loaded {len(malicious_map)} malicious users for Scenarios {target_scenarios}")

    sessions = []
    
    # Group by user to build sessions
    grouped_logon = logon_df.groupby('user')
    
    for user, group in tqdm(grouped_logon, desc="User Processing"):
        group = group.reset_index(drop=True)
        
        # Iterate through logon activities
        i = 0
        while i < len(group):
            if group.loc[i, 'activity'] == 'Logon':
                start_time = group.loc[i, 'date']
#                session_id = group.loc[i, 'id']
                
                # Find corresponding Logoff
                end_time = None
                j = i + 1
                while j < len(group):
                    if group.loc[j, 'activity'] == 'Logoff':
                        end_time = group.loc[j, 'date']
                        i = j # Skip to this logoff
                        break
                    elif group.loc[j, 'activity'] == 'Logon':
                        # New session started without logoff, assume close at new start or max 12h
                        end_time = group.loc[j, 'date']
                        i = j - 1 # Process this new logon next iteration
                        break
                    j += 1
                
                if end_time is None:
                    # If no logoff found, assume 8 hours duration
                    end_time = start_time + pd.Timedelta(hours=8)
                    i = len(group) # End of user logs
                
                # === FEATURE EXTRACTION FOR THIS SESSION ===
                
                # Helper to filter by user within the time slice
                def get_activity(df, u, s, e):
                    try:
                        slice_df = df[s:e]
                        return slice_df[slice_df['user'] == u]
                    except KeyError:
                        return pd.DataFrame(columns=df.columns)

                sess_email = get_activity(email_df, user, start_time, end_time)
                sess_http = get_activity(http_df, user, start_time, end_time)
                sess_file = get_activity(file_df, user, start_time, end_time)
                sess_device = get_activity(device_df, user, start_time, end_time)
                
                # -- Numerical Features (Agent 1) --
                duration = (end_time - start_time).total_seconds()
                
                # Exfiltration Indicators
                # 1. After Hours (7pm-7am or Weekend)
                is_weekend = 1 if start_time.weekday() >= 5 else 0
                is_after_hour = 1 if (start_time.hour >= 19 or start_time.hour < 7) else 0
                
                # 2. Email Exfiltration

                INTERNAL_DOMAIN = "dtaa.com"

                ext_emails = sess_email[~sess_email['to'].str.contains(INTERNAL_DOMAIN, na=False)]
                n_ext_emails = len(ext_emails)
                n_attachments = sess_email['attachments'].astype(str).apply(lambda x: 0 if x == 'nan' or x=='' else len(x.split(';'))).sum()
                total_email_size = sess_email['size'].sum()
                
                # 3. HTTP Exfiltration (Cloud Storage)
                cloud_keywords = ['dropbox', 'drive.google', 'mega', 'box.com', 'onedrive', 'wikileaks', 'pastebin']
                cloud_uploads = 0
                if not sess_http.empty:
                    cloud_uploads = sess_http['url'].apply(lambda x: 1 if any(k in str(x).lower() for k in cloud_keywords) else 0).sum()
                
                # 4. Device/File Exfiltration
                n_usb_connects = len(sess_device[sess_device['activity'].str.contains('Connect', case=False, na=False)])
                n_file_copies = len(sess_file)
                n_file_to_usb = 0
                if 'to_removable_media' in sess_file.columns:
                     n_file_to_usb = sess_file['to_removable_media'].astype(str).str.contains('True', case=False).sum()

                # -- Textual Data (For Agent 2) --
                # Extract specific content columns
                email_content_text = " ".join(sess_email['content'].dropna().astype(str).tolist()) if not sess_email.empty else ""
                http_url_text = " ".join(sess_http['url'].dropna().astype(str).tolist()) if not sess_http.empty else ""
                http_content_text = " ".join(sess_http['content'].dropna().astype(str).tolist()) if not sess_http.empty else ""
                file_names_text = " ".join(sess_file['filename'].dropna().astype(str).tolist()) if not sess_file.empty else ""
                file_content_text = " ".join(sess_file['content'].dropna().astype(str).tolist()) if not sess_file.empty else ""
                
#               --- 2. FAST LABELING CHECK ---
                label = 0
                if user in malicious_map:
                    for t_start, t_end in malicious_map[user]:
                        # Check if session overlaps with threat window
                        # Overlap logic: (StartA <= EndB) and (EndA >= StartB)
                        if (start_time <= t_end) and (end_time >= t_start):
                            label = 1
                            break

                sessions.append({
                    'user': user,
                    'start': start_time,
                    'end': end_time,
                    'duration': duration,
                    'is_weekend': is_weekend,
                    'is_after_hour': is_after_hour,
                    'emails_count': len(sess_email),
                    'ext_emails_count': n_ext_emails,
                    'attachments_count': n_attachments,
                    'total_email_size': total_email_size,
                    'email_content_text': email_content_text, # Agent 2
                    'http_count': len(sess_http),
                    'http_url_text': http_url_text, # Agent 2
                    'http_content_text': http_content_text, # Agent 2
                    'cloud_uploads_count': cloud_uploads,
                    'usb_connects_count': n_usb_connects,
                    'file_copies_count': n_file_copies,
                    'file_to_usb_count': n_file_to_usb,                  
                    'file_names_text': file_names_text, # Agent 2
                    'file_content_text': file_content_text, # Agent 2
                    'label': label
                })
            
            i += 1
            
    return pd.DataFrame(sessions)


### Dataset Preparation

In [6]:
FORCE_GENERATE = False # Set True to Overwrite existing file

if os.path.exists(CONFIG['processed_file']) and not FORCE_GENERATE:
    print(f"Found saved dataset: {CONFIG['processed_file']}")
    print("Loading data from file...")
    full_df = pd.read_csv(CONFIG['processed_file'])
    
    # POST-LOAD FIXES
    # 1. Restore Datetime objects
    full_df['start'] = pd.to_datetime(full_df['start'])
    full_df['end'] = pd.to_datetime(full_df['end'])
    
    # 2. Handle Text Columns (NaNs become empty strings)
    text_cols = ['email_content_text', 'http_url_text', 'http_content_text', 'file_names_text', 'file_content_text']
    full_df[text_cols] = full_df[text_cols].fillna("")
    
else:
    print("Starting Dataset Generation Pipeline")
    # 1. Load Raw Dataset Files
    raw_data = load_cert_data(CONFIG['dataset_path'])
    print("Datasets loaded")

    # 2. Creating sessions and extracting features
    full_df = create_sessions_and_features(raw_data)
    print("Sessions Created")
    
    # 3. Save
    print(f"Saving processed dataset to {CONFIG['processed_file']}...")
    full_df.to_csv(CONFIG['processed_file'], index=False)
    print("Save complete.")

print(f"Dataset Shape: {full_df.shape}")
print(f"Malicious Sessions: {full_df['label'].sum()}")
full_df.head()

Found saved dataset: ../DatasetsProcessed/data_exfiltration_content_dataset_updated.csv
Loading data from file...
Dataset Shape: (1948933, 21)
Malicious Sessions: 47


,user,start,end,duration,is_weekend,is_after_hour,emails_count,ext_emails_count,attachments_count,total_email_size,...,http_count,http_url_text,http_content_text,cloud_uploads_count,usb_connects_count,file_copies_count,file_to_usb_count,file_names_text,file_content_text,label
0,AAB0162,2010-01-04 07:41:00,2010-01-04 18:46:00,39900.0,0,0,9,1,2,2615549,...,95,http://barnesandnoble.com/Joseph_Szigeti/hubay...,"One of those, Testosterone, was filmed in 2003...",0,0,0,0,,,0
1,AAB0162,2010-01-05 07:46:00,2010-01-05 18:40:00,39240.0,0,0,9,1,3,2883730,...,95,http://pcmag.com/Bill_Ponsford/cripes/1996_Hav...,"In late 1996, the Justice Department opened a ...",0,0,0,0,,,0
2,AAB0162,2010-01-06 07:45:00,2010-01-06 18:55:00,40200.0,0,0,9,0,2,3028297,...,95,http://chase.com/Ediacara_biota/ediacara/Senax...,"Sinnock died in May 1947, before finishing the...",0,0,0,0,,,0
3,AAB0162,2010-01-07 07:45:00,2010-01-07 18:43:00,39480.0,0,0,9,2,1,2143731,...,95,http://foxsports.com/Psittacosaurus/psittacosa...,"In 1917-18, she was fitted with better rangefi...",0,0,0,0,,,0
4,AAB0162,2010-01-08 07:50:00,2010-01-08 18:41:00,39060.0,0,0,9,2,0,238896,...,95,http://pnc.com/Magnetosphere_of_Jupiter/rj/Oev...,"DePeyster also sent out Joseph Ainsse, a local...",0,0,0,0,,,0


### Preprocessing & Data Splitting

In [7]:
NUM_COLS = ['duration', 'is_weekend', 'is_after_hour', 'emails_count', 'ext_emails_count',
            'attachments_count', 'total_email_size', 'http_count', 'cloud_uploads_count',
            'usb_connects_count', 'file_copies_count', 'file_to_usb_count']

# ***CRITICAL FIX***: fit scaler but do NOT transform full_df in-place
scaler = MinMaxScaler()
scaler.fit(full_df[NUM_COLS])

# Split
benign_df = full_df[full_df['label'] == 0]
malicious_df = full_df[full_df['label'] == 1]

train_size = int(len(benign_df) * 0.8)
train_df = benign_df.iloc[:train_size]
test_benign_df = benign_df.iloc[train_size:]
test_df = pd.concat([test_benign_df, malicious_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Training Data (Normal Only): {len(train_df)}")
print(f"Testing Data (Mixed): {len(test_df)}")
print(f"  - Benign in test: {(test_df['label']==0).sum()}")
print(f"  - Malicious in test: {(test_df['label']==1).sum()}")

# SCALED tensors for Agent 1 only
X_train_num = torch.FloatTensor(scaler.transform(train_df[NUM_COLS])).to(CONFIG['device'])
X_test_num = torch.FloatTensor(scaler.transform(test_df[NUM_COLS])).to(CONFIG['device'])
y_test = test_df['label'].values

Training Data (Normal Only): 1559108
Testing Data (Mixed): 389825
  - Benign in test: 389778
  - Malicious in test: 47


### Single Agent Evaluation Function

In [8]:
def evaluate_single_agent(agent_name, scores, y_true):
    """Evaluate an individual agent's scoring with ROC-AUC and optimal threshold."""
    auc = roc_auc_score(y_true, scores)
    ap = average_precision_score(y_true, scores)
    print(f"\n{'='*60}")
    print(f"  {agent_name} — Individual Evaluation")
    print(f"{'='*60}")
    print(f"  ROC-AUC: {auc:.4f}")
    print(f"  Average Precision (AP): {ap:.6f}")

    # Find threshold that maximizes recall while keeping some precision
    precisions, recalls, thresholds_pr = precision_recall_curve(y_true, scores)

    # Find the threshold where recall >= 0.9 (catch 90%+ of malicious)
    target_recall = 0.90
    valid_mask = recalls[:-1] >= target_recall
    if valid_mask.any():
        # Among those with high recall, pick the one with best precision
        best_idx = np.argmax(precisions[:-1][valid_mask])
        actual_indices = np.where(valid_mask)[0]
        chosen_idx = actual_indices[best_idx]
        best_threshold = thresholds_pr[chosen_idx]
        best_precision = precisions[chosen_idx]
        best_recall = recalls[chosen_idx]
    else:
        # Fallback: just maximize F1
        f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
        chosen_idx = np.argmax(f1_scores)
        best_threshold = thresholds_pr[chosen_idx]
        best_precision = precisions[chosen_idx]
        best_recall = recalls[chosen_idx]

    print(f"  Optimal threshold (recall≥{target_recall:.0%}): {best_threshold:.6f}")
    print(f"  → Precision: {best_precision:.4f}, Recall: {best_recall:.4f}")

    preds = (scores >= best_threshold).astype(int)
    print(f"\n  Classification Report at optimal threshold:")
    print(classification_report(y_true, preds, target_names=['Benign', 'Malicious']))

    # How many of the actual malicious sessions are detected?
    n_mal = y_true.sum()
    n_detected = ((preds == 1) & (y_true == 1)).sum()
    n_fp = ((preds == 1) & (y_true == 0)).sum()
    print(f"  Malicious detected: {n_detected}/{n_mal}")
    print(f"  False positives: {n_fp}")

    return auc, scores

### Agent 1 - Contractive Autoencoder Model

In [9]:
class ContractiveAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(ContractiveAutoencoder, self).__init__()
        # Encoder
        self.fc1 = nn.Linear(input_dim, 32)
        self.fc2 = nn.Linear(32, 16)
        # Decoder
        self.fc3 = nn.Linear(16, 32)
        self.fc4 = nn.Linear(32, input_dim)
        
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Encoder
        h1 = self.relu(self.fc1(x))
        h2 = self.sigmoid(self.fc2(h1)) # Latent code
        
        # Decoder
        h3 = self.relu(self.fc3(h2))
        recon = self.sigmoid(self.fc4(h3)) # Output normalized 0-1
        return recon, h2

def contractive_loss(model, x, recon, h, lam):
    """
    Computes MSE + Contractive Penalty (Jacobian)
    """
    mse = nn.MSELoss()(recon, x)
    
    # Extract weights of the layer producing the latent code (fc2)
    W = model.fc2.weight
    
    # Calculate contraction penalty
    # h is shape (batch, latent_dim)
    dh = h * (1 - h) # Derivative of sigmoid
    
    # Sum of squares of weights
    w_sum = torch.sum(W**2, dim=1) # shape (latent_dim)
    w_sum = w_sum.unsqueeze(0) # shape (1, latent_dim)
    
    # Frobenius norm of Jacobian
    contractive_penalty = torch.sum(dh**2 * w_sum) / x.size(0)
    
    return mse + (lam * contractive_penalty), mse

### Training Agent 1

In [10]:
BATCH_SIZE = 256
EPOCHS = 120
LEARNING_RATE = 1e-4
LAMBDA_PENALTY = 1e-4
FORCE_RETRAIN_CAE = False

agent1_model = ContractiveAutoencoder(input_dim=len(NUM_COLS)).to(CONFIG['device'])
optimizer = optim.Adam(agent1_model.parameters(), lr=LEARNING_RATE)

if os.path.exists(CONFIG['agent1_path']) and not FORCE_RETRAIN_CAE:
    print(f"Found saved Agent 1 model at {CONFIG['agent1_path']}. Loading...")
    agent1_model.load_state_dict(
        torch.load(CONFIG['agent1_path'], map_location=CONFIG['device'])
    )
    agent1_model.eval()
else:
    print("Training CAE (Agent 1)...")
    train_loader = DataLoader(TensorDataset(X_train_num), batch_size=BATCH_SIZE, shuffle=True)
    loss_history = []
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch in train_loader:
            x_batch = batch[0]
            optimizer.zero_grad()
            recon, latent = agent1_model(x_batch)
            loss, mse = contractive_loss(agent1_model, x_batch, recon, latent, LAMBDA_PENALTY)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        loss_history.append(total_loss / len(train_loader))
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.6f}")

    torch.save(agent1_model.state_dict(), CONFIG['agent1_path'])
    print(f"Agent 1 CAE model saved to {CONFIG['agent1_path']}")
    plt.plot(loss_history)
    plt.title("Agent 1: CAE Training Loss")
    plt.show()

Found saved Agent 1 model at ../SavedModels/v10_agent1_cae.pth. Loading...


### Agent 1 Evaluation

In [11]:
agent1_model.eval()
loss_fn = nn.MSELoss(reduction='none')

# Training errors → used to compute dynamic threshold
with torch.no_grad():
    recon_train, _ = agent1_model(X_train_num)
    train_recon_errors = torch.mean(loss_fn(recon_train, X_train_num), dim=1).cpu().numpy()

# Dynamic threshold: mean + CAE_THRESHOLD_SIGMA * std of benign training errors
cae_threshold = train_recon_errors.mean() + CAE_THRESHOLD_SIGMA * train_recon_errors.std()
print(f"\nAgent 1 Dynamic Threshold (mean + {CAE_THRESHOLD_SIGMA}σ): {cae_threshold:.6f}")
print(f"  Training errors — mean: {train_recon_errors.mean():.6f}, std: {train_recon_errors.std():.6f}")

# Test errors
with torch.no_grad():
    recon_test, _ = agent1_model(X_test_num)
    reconstruction_errors = torch.mean(loss_fn(recon_test, X_test_num), dim=1).cpu().numpy()


Agent 1 Dynamic Threshold (mean + 2.0σ): 0.000179
  Training errors — mean: 0.000021, std: 0.000079


### Initialize Blackboard & Agent 1 Writes

In [12]:
print("\n" + "="*60)
print("  INITIALIZING SHARED BLACKBOARD")
print("="*60)

# Create blackboard for TEST data
bb_test = Blackboard(n_samples=len(test_df))

# Agent 1 writes to the blackboard
bb_test.write('agent1', 'scores', reconstruction_errors,
              msg=f"Agent1 wrote {len(reconstruction_errors)} reconstruction error scores")
bb_test.write('agent1', 'dynamic_threshold', cae_threshold,
              msg=f"Agent1 set dynamic threshold = {cae_threshold:.6f}")

cae_high_flag = reconstruction_errors > cae_threshold
n_flagged = cae_high_flag.sum()
bb_test.write('agent1', 'high_flag', cae_high_flag,
              msg=f"Agent1 flagged {n_flagged} sessions ({n_flagged/len(test_df)*100:.2f}%) above threshold")

# --- Agent 1 Evaluation ---
agent1_auc, _ = evaluate_single_agent("Agent 1 (CAE)", reconstruction_errors, y_test)

# Also create a blackboard for TRAINING data (for orchestrator training)
bb_train = Blackboard(n_samples=len(train_df))
train_cae_high_flag = train_recon_errors > cae_threshold
bb_train.write('agent1', 'scores', train_recon_errors)
bb_train.write('agent1', 'dynamic_threshold', cae_threshold)
bb_train.write('agent1', 'high_flag', train_cae_high_flag)


  INITIALIZING SHARED BLACKBOARD
  [Blackboard] Agent1 wrote 389825 reconstruction error scores
  [Blackboard] Agent1 set dynamic threshold = 0.000179
  [Blackboard] Agent1 flagged 8158 sessions (2.09%) above threshold

  Agent 1 (CAE) — Individual Evaluation
  ROC-AUC: 0.8701
  Average Precision (AP): 0.000625
  Optimal threshold (recall≥90%): 0.000014
  → Precision: 0.0004, Recall: 0.9149

  Classification Report at optimal threshold:
              precision    recall  f1-score   support

      Benign       1.00      0.72      0.84    389778
   Malicious       0.00      0.91      0.00        47

    accuracy                           0.72    389825
   macro avg       0.50      0.82      0.42    389825
weighted avg       1.00      0.72      0.84    389825

  Malicious detected: 43/47
  False positives: 108931


### Agent 2 - DistilBERT Threat Classifier (NLP)

In [13]:
AGENT2_BATCH_SIZE = 16

class DistilBertAgent:
    """Agent 2: MPNet Scanner + Fine-tuned DistilBERT Classifier.
    
    Pipeline per session:
      1. Combine all text columns into one string
      2. Chunk the text and embed with MPNet
      3. Find most suspicious chunk via cosine similarity to threat anchors
      4. Classify that chunk with DistilBERT (Normal / Job Search / Data Exfiltration)
      5. Use threat-class probability as anomaly score (0-1)
    """

    def __init__(self, scanner_model_name, model_dir, device):
        print(f"Initializing MPNet Scanner ({scanner_model_name})...")
        self.scanner = SentenceTransformer(scanner_model_name, device=device)
        self.device = device
        self.model_dir = model_dir
        self.tokenizer = None
        self.classifier = None

        # Anchor texts for the scanner
        self.job_anchor_text = (
            "I want to quit my job. I am applying for a new position. "
            "looking for a new job, applying for positions, sending resume, interview, recruiter, linkedin, "
            "glassdoor, monster.com, taleo, careerbuilder, layoff rumors, downsizing anxiety, severance package, "
            "Boeing, Lockheed Martin, Raytheon, Northrop Grumman, mushrooms, suillus spraguei, bad date email signature, "
            "updating cv, reference check, salary negotiation"
        )

        self.exfil_anchor_text = (
            "I am stealing company secrets. I am uploading confidential files to a personal account. "
            "stealing proprietary data, uploading confidential files to cloud storage, dropbox, google drive, "
            "wikileaks, julian assange, the real story about dtaa, jainism, anekantavada, gandhi, "
            "m249 saw, fuzes, hazard analysis, strategic systems program, hms lion, uranus operation, cockatoo, "
            "copying files to usb, removable media, thumb drive, tor browser, encryption, steganography"
        )

        self.anchor_job_vec = self.scanner.encode(self.job_anchor_text, convert_to_tensor=True)
        self.anchor_exfil_vec = self.scanner.encode(self.exfil_anchor_text, convert_to_tensor=True)

    def load(self):
        """Load the fine-tuned DistilBERT model and tokenizer."""
        print(f"Loading Fine-Tuned DistilBERT from {self.model_dir}...")
        if not os.path.exists(self.model_dir):
            raise FileNotFoundError(f"Model directory not found: {self.model_dir}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.classifier = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.classifier.to(self.device)
        self.classifier.eval()
        print("DistilBERT model loaded successfully.")

    def _prepare_texts(self, df):
        """Combine all text columns into a single text per session."""
        text_cols = ['email_content_text', 'http_url_text', 'http_content_text',
                     'file_names_text', 'file_content_text']
        labels = ['EMAIL', 'HTTP URL', 'HTTP CONTENT', 'FILE NAMES', 'FILE CONTENT']
        parts = []
        for col, label in zip(text_cols, labels):
            series = df[col].fillna('').astype(str)
            mask = series.str.strip().ne('')
            labeled = (label + ': ' + series).where(mask, '')
            parts.append(labeled)
        combined = parts[0]
        for p in parts[1:]:
            both_nonempty = combined.ne('') & p.ne('')
            combined = combined + both_nonempty.map({True: ' | ', False: ''}) + p
        return combined.tolist()

    def _get_best_snippet_batch(self, texts):
        """For each text, chunk it and find the most suspicious chunk via anchor similarity."""
        batch_best_snippets = []
        for text in texts:
            if pd.isna(text) or text.strip() == "":
                batch_best_snippets.append("")
                continue

            chunk_size = 800
            overlap = 150
            chunks = [text[i : i + chunk_size] for i in range(0, len(text), chunk_size - overlap)]
            if not chunks:
                chunks = [text[:512]]
            chunks = chunks[:100]

            chunk_embeddings = self.scanner.encode(chunks, convert_to_tensor=True, show_progress_bar=False)

            scores_job = util.cos_sim(chunk_embeddings, self.anchor_job_vec).squeeze()
            scores_exfil = util.cos_sim(chunk_embeddings, self.anchor_exfil_vec).squeeze()

            if scores_job.ndim == 0:
                scores_job = scores_job.unsqueeze(0)
            if scores_exfil.ndim == 0:
                scores_exfil = scores_exfil.unsqueeze(0)

            max_scores, _ = torch.max(torch.stack([scores_job, scores_exfil]), dim=0)
            best_idx = torch.argmax(max_scores).item()
            batch_best_snippets.append(chunks[best_idx])

        return batch_best_snippets

    def _predict_batch(self, snippets):
        """Classify snippets using the DistilBERT classifier."""
        valid_snippets = [s if s.strip() else "empty session" for s in snippets]
        inputs = self.tokenizer(valid_snippets, return_tensors="pt", padding=True,
                                truncation=True, max_length=512).to(self.device)
        with torch.no_grad():
            outputs = self.classifier(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        preds = torch.argmax(probs, dim=-1).cpu().numpy()
        # Threat probability = sum of class 1 (job search) + class 2 (data exfiltration)
        threat_probs = (probs[:, 1] + probs[:, 2]).cpu().numpy()
        return preds, threat_probs

    def predict_score(self, df, batch_size=16):
        """Compute anomaly scores: higher = more suspicious."""
        print("Agent 2: Preparing texts...")
        texts = self._prepare_texts(df)
        has_text = [bool(t.strip()) for t in texts]

        scores = np.zeros(len(texts))
        text_indices = [i for i, h in enumerate(has_text) if h]

        if not text_indices:
            return scores

        text_batch_all = [texts[i] for i in text_indices]
        all_threat_probs = []

        print(f"Agent 2: Processing {len(text_batch_all):,} sessions with text content...")
        for i in tqdm(range(0, len(text_batch_all), batch_size), desc="Agent 2 Processing"):
            batch_texts = text_batch_all[i : i + batch_size]

            # Step 1: Find most suspicious snippet per text
            snippets = self._get_best_snippet_batch(batch_texts)

            # Step 2: Classify snippets with DistilBERT
            preds, threat_probs = self._predict_batch(snippets)
            all_threat_probs.extend(threat_probs)

        # Assign threat probabilities as anomaly scores
        for idx, threat_prob in zip(text_indices, all_threat_probs):
            scores[idx] = threat_prob

        return scores

    def predict_score_with_blackboard(self, df, blackboard, boost_multiplier=1.5, batch_size=16):
        """
        Blackboard-aware scoring:
        - Reads Agent 1's high_flag
        - Boosts score by `boost_multiplier` for sessions flagged by CAE
        """
        print("Agent 2: Computing DistilBERT threat scores...")
        base_scores = self.predict_score(df, batch_size=batch_size)

        # Read CAE flag from blackboard
        cae_high_flag = blackboard.get_flag('agent1', 'high_flag')

        if cae_high_flag is not None and cae_high_flag.any():
            n_boosted = cae_high_flag.sum()
            boosted_scores = np.where(cae_high_flag,
                                      np.clip(base_scores * boost_multiplier, 0, 1),
                                      base_scores)
            blackboard.write('agent2', 'scores', boosted_scores,
                             msg=f"Agent2 boosted {n_boosted} sessions by {boost_multiplier}x "
                                 f"based on Agent1 high_flag")
            blackboard.write('agent2', 'base_scores', base_scores,
                             msg=f"Agent2 also stored unboosted base scores for comparison")
            return boosted_scores
        else:
            blackboard.write('agent2', 'scores', base_scores,
                             msg="Agent2 ran without blackboard boost (no Agent1 flags)")
            blackboard.write('agent2', 'base_scores', base_scores)
            return base_scores


# --- Agent 2 Loading ---
agent2 = DistilBertAgent(
    scanner_model_name=CONFIG['agent2_scanner_model'],
    model_dir=CONFIG['agent2_model_dir'],
    device=CONFIG['device']
)
agent2.load()


Initializing MPNet Scanner (all-mpnet-base-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading Fine-Tuned DistilBERT from ../SavedModels/final_insider_threat_model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT model loaded successfully.


### Agent 2 DistilBERT Scoring + Evaluation

In [14]:
# Score TEST data with blackboard boost
semantic_scores = agent2.predict_score_with_blackboard(
    test_df, bb_test, boost_multiplier=NLP_BOOST_MULTIPLIER, batch_size=AGENT2_BATCH_SIZE
)

# Write NLP high flag to blackboard
nlp_threshold = np.percentile(semantic_scores, NLP_FLAG_PERCENTILE)
nlp_high_flag = semantic_scores > nlp_threshold
bb_test.write('agent2', 'high_flag', nlp_high_flag,
              msg=f"Agent2 flagged {nlp_high_flag.sum()} sessions above "
                  f"P{NLP_FLAG_PERCENTILE} threshold = {nlp_threshold:.4f}")

agent2_auc, _ = evaluate_single_agent("Agent 2 (DistilBERT + Boost)", semantic_scores, y_test)

# Score TRAINING data with blackboard boost (for orchestrator training)
train_semantic_scores = agent2.predict_score_with_blackboard(
    train_df, bb_train, boost_multiplier=NLP_BOOST_MULTIPLIER, batch_size=AGENT2_BATCH_SIZE
)
train_nlp_high_flag = train_semantic_scores > nlp_threshold
bb_train.write('agent2', 'high_flag', train_nlp_high_flag)


Agent 2: Computing DistilBERT threat scores...
Agent 2: Preparing texts...
Agent 2: Processing 363,108 sessions with text content...


Agent 2 Processing:   0%|          | 0/22695 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Agent 3 - Temporal/Behavioral Baseline Agent with Blackboard Context

In [ ]:
FORCE_RETRAIN_TEMPORAL = True

class TemporalBaselineAgent:
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols
        self.user_means = None
        self.user_stds = None
        self.global_mean = None
        self.global_std = None

    def fit(self, train_df):
        print("Agent 3: Building Historical User Baselines (on RAW features)...")
        self.global_mean = train_df[self.feature_cols].astype(np.float64).mean()
        self.global_std = train_df[self.feature_cols].astype(np.float64).std().replace(0, 1e-6)

        grouped = train_df.groupby('user')[self.feature_cols]
        self.user_means = grouped.mean()
        self.user_stds = grouped.std()

        for col in self.feature_cols:
            self.user_means[col] = self.user_means[col].fillna(self.global_mean[col])
            self.user_stds[col] = (
                self.user_stds[col].fillna(self.global_std[col]).replace(0, 1e-6)
            )

        print(f"Agent 3: Baselines built for {len(self.user_means)} unique users.")

    def predict_score(self, test_df):
        """Base per-user Z-score deviation (no blackboard)."""
        means_dict = self.user_means.to_dict('index')
        stds_dict = self.user_stds.to_dict('index')

        x = test_df[self.feature_cols].values.astype(np.float64)
        means = np.zeros(x.shape, dtype=np.float64)
        stds = np.ones(x.shape, dtype=np.float64)

        for i, user in enumerate(test_df['user']):
            if user in means_dict:
                means[i] = list(means_dict[user].values())
                stds[i] = list(stds_dict[user].values())
            else:
                means[i] = self.global_mean.values
                stds[i] = self.global_std.values

        stds = np.maximum(stds, 1e-6)
        z_scores = (x - means) / stds
        z_scores = np.clip(z_scores, -10.0, 10.0)
        temporal_anomaly_scores = np.sqrt(np.mean(z_scores**2, axis=1))
        return temporal_anomaly_scores

    def _compute_global_z_scores(self, test_df):
        """
        Compute Z-scores using GLOBAL baselines (all users' mean/std).
        Used for deeper context when upstream agents flag a session.
        This is the 'wider 30-day window' equivalent — comparing against
        the entire population instead of just the individual user.
        """
        x = test_df[self.feature_cols].values.astype(np.float64)
        global_means = np.tile(self.global_mean.values, (len(x), 1))
        global_stds = np.tile(self.global_std.values, (len(x), 1))
        global_stds = np.maximum(global_stds, 1e-6)

        z_scores = (x - global_means) / global_stds
        z_scores = np.clip(z_scores, -10.0, 10.0)
        return np.sqrt(np.mean(z_scores**2, axis=1))

    def predict_score_with_blackboard(self, test_df, blackboard):
        """
        Blackboard-aware scoring:
        - Reads Agent 1 and Agent 2 high_flag
        - For sessions flagged by either upstream agent, also compute
          global Z-scores and take max(per-user, global)
        """
        print("Agent 3: Computing per-user Z-Score Deviations...")
        base_scores = self.predict_score(test_df)

        # Read upstream flags
        cae_high = blackboard.get_flag('agent1', 'high_flag')
        nlp_high = blackboard.get_flag('agent2', 'high_flag')

        # Sessions flagged by either upstream agent
        upstream_flagged = cae_high | nlp_high
        n_flagged = upstream_flagged.sum()

        if n_flagged > 0:
            print(f"Agent 3: {n_flagged} sessions flagged by upstream agents — computing global baselines...")
            global_scores = self._compute_global_z_scores(test_df)
            # Use max(per-user, global) for flagged sessions → deeper context
            enhanced_scores = np.where(upstream_flagged,
                                       np.maximum(base_scores, global_scores),
                                       base_scores)
            blackboard.write('agent3', 'scores', enhanced_scores,
                             msg=f"Agent3 enhanced {n_flagged} sessions with global baselines "
                                 f"(max of per-user and global Z-scores)")
            blackboard.write('agent3', 'base_scores', base_scores,
                             msg=f"Agent3 stored base per-user scores for comparison")
            return enhanced_scores
        else:
            print("Agent 3: No upstream flags — using standard per-user scoring.")
            blackboard.write('agent3', 'scores', base_scores,
                             msg="Agent3 ran without enhancement (no upstream flags)")
            blackboard.write('agent3', 'base_scores', base_scores)
            return base_scores

    def save(self, path):
        data = {
            'user_means': self.user_means, 'user_stds': self.user_stds,
            'global_mean': self.global_mean, 'global_std': self.global_std
        }
        with open(path, 'wb') as f:
            pickle.dump(data, f)

    def load(self, path):
        with open(path, 'rb') as f:
            data = pickle.load(f)
        self.user_means = data['user_means']
        self.user_stds = data['user_stds']
        self.global_mean = data['global_mean']
        self.global_std = data['global_std']


# --- Agent 3 Training ---
agent3 = TemporalBaselineAgent(feature_cols=NUM_COLS)
if os.path.exists(CONFIG['agent3_path']) and not FORCE_RETRAIN_TEMPORAL:
    print(f"Loading Agent 3 from {CONFIG['agent3_path']}...")
    agent3.load(CONFIG['agent3_path'])
else:
    print("Training Agent 3...")
    agent3.fit(train_df)
    os.makedirs(os.path.dirname(CONFIG['agent3_path']), exist_ok=True)
    agent3.save(CONFIG['agent3_path'])
    print(f"Agent 3 saved to {CONFIG['agent3_path']}")

### Agent 3 Evaluation

In [ ]:
temporal_scores = agent3.predict_score_with_blackboard(test_df, bb_test)
agent3_auc, _ = evaluate_single_agent("Agent 3 (Temporal + Blackboard Context)",
                                      temporal_scores, y_test)

# Score training data
train_temporal_scores = agent3.predict_score_with_blackboard(train_df, bb_train)

### Blackboard Summary

In [ ]:
bb_test.summary()

### Agent 4 - Orchestrator & Results

#### Build Feature Matrix for Orchestrator with Blackboard Evidence

In [ ]:
def build_orchestrator_features(score_cae, score_nlp, score_temp):
    """Base orchestrator feature matrix (from V8.1)."""
    features = np.column_stack([
        score_cae,
        score_nlp,
        score_temp,
        score_cae * score_temp,
        score_cae * score_nlp,
        score_temp * score_nlp,
        np.max([score_cae, score_temp, score_nlp], axis=0),
        np.std([score_cae, score_temp, score_nlp], axis=0),
    ])
    return features

def build_orchestrator_features_with_blackboard(bb):
    """
    Build enriched orchestrator features including blackboard evidence.
    Adds: CAE flag (binary), upstream consensus count (0-3).
    """
    score_cae = bb.read('agent1', 'scores')
    score_nlp = bb.read('agent2', 'scores')
    score_temp = bb.read('agent3', 'scores')

    # Standard features
    base_features = build_orchestrator_features(score_cae, score_nlp, score_temp)

    # --- NEW: Blackboard evidence features ---
    cae_flag = bb.get_flag('agent1', 'high_flag').astype(float)
    nlp_flag = bb.get_flag('agent2', 'high_flag').astype(float)

    # Temporal flag: top 10% of temporal scores
    temp_threshold = np.percentile(score_temp, 90)
    temp_flag = (score_temp > temp_threshold).astype(float)

    # Consensus: how many agents independently flagged this session?
    upstream_agreement = cae_flag + nlp_flag + temp_flag

    enriched_features = np.column_stack([
        base_features,
        cae_flag,              # Binary: did CAE flag this?
        upstream_agreement,    # Count: how many agents flagged? (0-3)
    ])

    return enriched_features


FEATURE_NAMES = [
    'CAE_score', 'Semantic_score', 'Temporal_score',
    'CAE×Temporal', 'CAE×Semantic', 'Temporal×Semantic',
    'Max_score', 'Score_std',
    'CAE_flag', 'Upstream_agreement'  # NEW blackboard evidence
]

# Build training features
X_orch_train = build_orchestrator_features_with_blackboard(bb_train)

# Build test features
X_orch_test = build_orchestrator_features_with_blackboard(bb_test)

print(f"\nOrchestrator training features: {X_orch_train.shape}")
print(f"Orchestrator test features:     {X_orch_test.shape}")
print(f"Feature names: {FEATURE_NAMES}")

### Agent 4 — Isolation Forest Orchestrator (Semi-Supervised)

In [ ]:
# The IF learns the distribution of agent-score patterns from BENIGN training
# data. At test time, sessions whose agent-score combination deviates from
# the learned benign distribution → anomalous (potential exfiltration).
#
# Why this works:
#   - A benign session has LOW CAE error, LOW semantic distance, LOW temporal Z-score
#   - A malicious session has UNUSUAL combinations (high on multiple agents)
#   - The IF learns the shape of "normal" in the multi-dimensional score space
#   - Non-linear: it can learn that high-CAE + high-Temporal is MORE suspicious
#     than a linear sum would suggest

FORCE_RETRAIN_ORCHESTRATOR = True

if os.path.exists(CONFIG['agent4_path']) and not FORCE_RETRAIN_ORCHESTRATOR:
    print(f"Loading Agent 4 Orchestrator from {CONFIG['agent4_path']}...")
    with open(CONFIG['agent4_path'], 'rb') as f:
        orchestrator_if = pickle.load(f)
else:
    print("\nTraining Agent 4: Isolation Forest Orchestrator...")
    print(f"  Training on {len(X_orch_train):,} benign score patterns")
    print(f"  Feature dimensions: {X_orch_train.shape[1]} ({', '.join(FEATURE_NAMES)})")

    orchestrator_if = IsolationForest(
        n_estimators=300,
        contamination='auto',
        max_samples='auto',
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    orchestrator_if.fit(X_orch_train)

    os.makedirs(os.path.dirname(CONFIG['agent4_path']), exist_ok=True)
    with open(CONFIG['agent4_path'], 'wb') as f:
        pickle.dump(orchestrator_if, f)
    print(f"  Agent 4 Orchestrator saved to {CONFIG['agent4_path']}")

# Score test data with IF
if_raw_scores = orchestrator_if.score_samples(X_orch_test)
final_scores_if = -if_raw_scores  # Negate: higher = more anomalous

print(f"\n  IF score range: [{final_scores_if.min():.4f}, {final_scores_if.max():.4f}]")
print(f"  IF score mean (benign): {final_scores_if[y_test == 0].mean():.4f}")
print(f"  IF score mean (malicious): {final_scores_if[y_test == 1].mean():.4f}")

### Agent 4 (IF) Evaluation

In [ ]:
if_auc, _ = evaluate_single_agent("Agent 4 (IF Orchestrator + Blackboard Evidence)",
                                  final_scores_if, y_test)

### Linear Fusion Baseline (for comparison)

In [ ]:
def normalize_rank(arr):
    return rankdata(arr) / len(arr)

norm_cae = normalize_rank(reconstruction_errors)
norm_nlp = normalize_rank(semantic_scores)
norm_temp = normalize_rank(temporal_scores)

w_cae = 0.40
w_temp = 0.30
w_nlp = 0.30

final_scores_linear = (w_cae * norm_cae) + (w_temp * norm_temp) + (w_nlp * norm_nlp)
linear_auc = roc_auc_score(y_test, final_scores_linear)
linear_ap = average_precision_score(y_test, final_scores_linear)

print(f"\n{'='*60}")
print(f"  Linear Fusion Baseline")
print(f"{'='*60}")
print(f"  Linear ROC-AUC: {linear_auc:.4f}")
print(f"  Linear Average Precision: {linear_ap:.4f}")

### Optimal Threshold Selection for IF Orchestrator

### Threshold Analysis — IF Orchestrator

In [ ]:
print(f"\n{'='*60}")
print(f"  Threshold Analysis — IF Orchestrator")
print(f"{'='*60}")

precisions, recalls, thresholds_pr = precision_recall_curve(y_test, final_scores_if)

for target in [1.0, 0.95, 0.90, 0.85, 0.80]:
    valid_mask = recalls[:-1] >= target
    if valid_mask.any():
        valid_precisions = precisions[:-1][valid_mask]
        best_local_idx = np.argmax(valid_precisions)
        actual_indices = np.where(valid_mask)[0]
        chosen_idx = actual_indices[best_local_idx]
        t = thresholds_pr[chosen_idx]
        p = precisions[chosen_idx]
        r = recalls[chosen_idx]
        preds_temp = (final_scores_if >= t).astype(int)
        n_mal = y_test.sum()
        n_detected = ((preds_temp == 1) & (y_test == 1)).sum()
        n_fp = ((preds_temp == 1) & (y_test == 0)).sum()
        print(f"\n  Target recall ≥ {target:.0%}:")
        print(f"    Threshold: {t:.6f}, Precision: {p:.4f}, Recall: {r:.4f}")
        print(f"    Malicious detected: {n_detected}/{n_mal}, False positives: {n_fp}")

# Select best threshold: 100% recall if possible, else 90%
best_target = 1.0
valid_mask = recalls[:-1] >= best_target
if not valid_mask.any():
    best_target = 0.90
    valid_mask = recalls[:-1] >= best_target

if valid_mask.any():
    valid_precisions = precisions[:-1][valid_mask]
    best_local_idx = np.argmax(valid_precisions)
    actual_indices = np.where(valid_mask)[0]
    chosen_idx = actual_indices[best_local_idx]
    threshold = thresholds_pr[chosen_idx]
else:
    threshold = 0.0

predictions = (final_scores_if >= threshold).astype(int)

n_detected = ((predictions == 1) & (y_test == 1)).sum()
n_mal = y_test.sum()
print(f"\n  *** Selected threshold: {threshold:.6f} ***")
print(f"  *** Malicious detected: {n_detected}/{n_mal} ***")

print(f"\n{'='*60}")
print(f"  Final Classification Report (IF Orchestrator + Blackboard)")
print(f"{'='*60}")
print(classification_report(y_test, predictions, target_names=['Benign', 'Malicious']))
print(f"ROC-AUC Score: {if_auc:.4f}")

### Score Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
sns.histplot(final_scores_if[y_test == 0], color='blue', label='Benign',
             kde=True, stat="density", bins=50, ax=ax1)
sns.histplot(final_scores_if[y_test == 1], color='red', label='Malicious',
             kde=True, stat="density", bins=50, ax=ax1)
ax1.axvline(threshold, color='green', linestyle='--', label=f"Threshold ({threshold:.4f})")
ax1.set_title("IF Orchestrator + Blackboard — Score Distribution")
ax1.set_xlabel("Anomaly Score (higher = more anomalous)")
ax1.set_ylabel("Density")
ax1.legend()

ax2 = axes[1]
sns.histplot(final_scores_linear[y_test == 0], color='blue', label='Benign',
             kde=True, stat="density", bins=50, ax=ax2)
sns.histplot(final_scores_linear[y_test == 1], color='red', label='Malicious',
             kde=True, stat="density", bins=50, ax=ax2)
ax2.set_title("Linear Fusion Score Distribution (Baseline)")
ax2.set_xlabel("Fused Rank Score")
ax2.set_ylabel("Density")
ax2.legend()

plt.tight_layout()
plt.show()

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Malicious'],
            yticklabels=['Normal', 'Malicious'])
plt.title("Confusion Matrix — Blackboard MAS V10")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

for name, scores, auc_val in [
    ("Agent 1 (CAE)", reconstruction_errors, agent1_auc),
    ("Agent 2 (Semantic + Boost)", semantic_scores, agent2_auc),
    ("Agent 3 (Temporal + Context)", temporal_scores, agent3_auc),
    ("Agent 4 (IF + Blackboard)", final_scores_if, if_auc),
    ("Linear Fusion Baseline", final_scores_linear, linear_auc),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})")

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison — Blackboard MAS V10")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### Agent Comparison Summary

In [ ]:
print(f"\n{'='*60}")
print(f"  FINAL SUMMARY — Multi-Agent System V10 (Shared Blackboard)")
print(f"{'='*60}")
print(f"  Agent 1 (CAE)                    ROC-AUC: {agent1_auc:.4f}")
print(f"  Agent 2 (Semantic + Boost)       ROC-AUC: {agent2_auc:.4f}")
print(f"  Agent 3 (Temporal + Context)     ROC-AUC: {agent3_auc:.4f}")
print(f"  Agent 4 (IF + Blackboard)        ROC-AUC: {if_auc:.4f}")
print(f"  Linear Fusion Baseline           ROC-AUC: {linear_auc:.4f}")
print(f"")
print(f"  Blackboard Messages Exchanged: {len(bb_test.get_messages())}")
print(f"  CAE Dynamic Threshold: {cae_threshold:.6f}")
print(f"  NLP Boost Multiplier: {NLP_BOOST_MULTIPLIER}")
print(f"  Sessions CAE-flagged: {bb_test.get_flag('agent1', 'high_flag').sum()}")
print(f"  Sessions NLP-flagged: {bb_test.get_flag('agent2', 'high_flag').sum()}")
print(f"{'='*60}")